**02_preprocessing_pipeline.ipynb: Leakage-Safe Preprocessing & Data Pipeline Verification**

**Multimodal Financial Fraud / Risk Detector**  
This notebook verifies the end-to-end Phase 3 data pipeline: chronological splitting, train-only fitting, imputation, scaling, vocabulary indexing, and PyTorch `FraudDataset` & `DataLoader` mini-batch extraction.

```
PHASE 3 WORKFLOW
│
├── 3.1 Chronological Train / Validation / Test Splitting
├── 3.2 Numerical Preprocessing (Log Transform, Train Median Imputation, Standardization)
├── 3.3 Categorical Preprocessing (Train Vocabularies, Rare Frequency Thresholding, <UNK>)
├── 3.4 Leakage Prevention Audit (Train-only parameters, Zero Lookahead)
├── 3.5 Feature Selection (Identifier exclusion, Zero-variance removal)
├── 3.6 PyTorch Dataset (FraudDataset tensor packing)
├── 3.7 PyTorch DataLoader (Train, Validation, and Test loaders)
└── 3.8 Pipeline Verification (Batch shapes, dtypes, range inspection)
```


**1. Environment Setup & Component Imports**

We import `TabularPreprocessor` from `src.preprocessing` and `FraudDataset`, `create_dataloaders` from `src.dataset`.


In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from src.preprocessing import (
    load_and_merge_data,
    temporal_train_val_test_split,
    TabularPreprocessor
)
from src.dataset import FraudDataset, create_dataloaders

print("Phase 3 modules successfully imported.")
print(f"PyTorch Version: {torch.__version__}")


Phase 3 modules successfully imported.
PyTorch Version: 2.14.0+cpu


---
**3.1 Chronological Train / Validation / Test Splitting**

Per Phase 1 & 2 recommendations, transactions are sorted by `TransactionDT` and strictly partitioned into:
- **Train**: Historical transactions (70%)
- **Validation**: Intermediate transactions (15%)
- **Test**: Future transactions (15%)


In [2]:
NROWS = 100_000  # Set to None for the complete 590,540 rows
data_dir = PROJECT_ROOT / "data" / "raw"

# Load merged transactions and identities
df = load_and_merge_data(data_dir=data_dir, split="train", nrows=NROWS, reduce_memory=True, verbose=True)

# Execute strict temporal split
train_df, val_df, test_df = temporal_train_val_test_split(df, val_ratio=0.15, test_ratio=0.15)

# Inspect distribution shift across time
splits = [("Train", train_df), ("Val", val_df), ("Test", test_df)]
print("\n=== Temporal Split & Fraud Rate Summary ===")
for name, s_df in splits:
    fraud_rate = s_df["isFraud"].mean() * 100
    t_min = s_df["TransactionDT"].min()
    t_max = s_df["TransactionDT"].max()
    print(f"{name:5s} Split: {len(s_df):,} rows | DT: [{t_min:,} - {t_max:,}] | Fraud Rate: {fraud_rate:.2f}%")


2026-09-30 23:13:52,012 - INFO - Loading train_transaction.csv (nrows=100000)...
2026-09-30 23:13:56,573 - INFO - Memory reduced from 300.60 MB to 155.16 MB (48.4% reduction)
2026-09-30 23:13:56,574 - INFO - Successfully loaded train_transaction.csv with shape: (100000, 394)
2026-09-30 23:13:56,576 - INFO - Loading train_identity.csv (nrows=100000)...
2026-09-30 23:13:56,995 - INFO - Memory reduced from 31.28 MB to 22.13 MB (29.3% reduction)
2026-09-30 23:13:56,996 - INFO - Successfully loaded train_identity.csv with shape: (100000, 41)
2026-09-30 23:13:57,003 - INFO - Merging train transaction (100000 rows) and identity (100000 rows) on TransactionID...
2026-09-30 23:13:57,288 - INFO - Merged dataset shape: (100000, 434). Identity match rate: 40.28% (40283/100000)
2026-09-30 23:13:57,394 - INFO - Sorting dataset by 'TransactionDT' for strict chronological temporal splitting...
2026-09-30 23:13:57,833 - INFO - Temporal Split Completed Successfully:
2026-09-30 23:13:57,835 - INFO -   [T


=== Temporal Split & Fraud Rate Summary ===
Train Split: 70,000 rows | DT: [86,400 - 1,564,844] | Fraud Rate: 2.68%
Val   Split: 15,000 rows | DT: [1,564,850 - 1,804,023] | Fraud Rate: 2.51%
Test  Split: 15,000 rows | DT: [1,804,027 - 2,006,364] | Fraud Rate: 2.03%


---
**3.2 to 3.5 Leakage-Safe Fitting & Feature Selection**

We initialize `TabularPreprocessor` and fit it **strictly on `train_df`**:
- **3.2 Numerical Pipeline**: Applies `np.log1p(TransactionAmt)`, fits train-set medians for imputation, and fits `StandardScaler`.
- **3.3 Categorical Pipeline**: Builds vocabularies using frequency threshold `min_freq = 10` on train set; assigns index `0` to `<UNK>`/rare/missing.
- **3.4 Leakage Prevention**: No validation or test set statistics are accessed during fitting.
- **3.5 Feature Selection**: Excludes `TransactionID`, `TransactionDT`, and drops zero-variance numerical features.


In [3]:
# Initialize and fit strictly on train_df
preprocessor = TabularPreprocessor(
    min_freq=10,
    log_transform_amt=True,
    drop_zero_variance=True
)

preprocessor.fit(train_df)

print("=== Preprocessor Fitting Summary ===")
print(f"Numerical features fitted:   {len(preprocessor.fitted_numerical_cols_)}")
print(f"Categorical features fitted: {len(preprocessor.fitted_categorical_cols_)}")
print(f"Dropped zero-variance cols:  {preprocessor.dropped_cols_}")

# Transform all three splits using train-fitted parameters
X_num_train, X_cat_train, y_train = preprocessor.transform(train_df)
X_num_val, X_cat_val, y_val = preprocessor.transform(val_df)
X_num_test, X_cat_test, y_test = preprocessor.transform(test_df)

print("\nTransformed Array Dimensions:")
print(f"  Train: Numerical {X_num_train.shape}, Categorical {X_cat_train.shape}, Labels {y_train.shape}")
print(f"  Val:   Numerical {X_num_val.shape}, Categorical {X_cat_val.shape}, Labels {y_val.shape}")
print(f"  Test:  Numerical {X_num_test.shape}, Categorical {X_cat_test.shape}, Labels {y_test.shape}")


2026-09-30 23:13:57,876 - INFO - Fitting TabularPreprocessor strictly on training data (Leakage-Safe)...
2026-09-30 23:13:59,363 - INFO - Dropping 1 zero-variance numerical features: ['V107']
2026-09-30 23:13:59,367 - INFO - Selected 381 numerical and 49 categorical features.
2026-09-30 23:14:00,603 - INFO - Successfully fitted StandardScaler on imputed training numericals.
2026-09-30 23:14:01,424 - INFO - Built vocabularies for 49 categorical features (min_freq=10).


=== Preprocessor Fitting Summary ===
Numerical features fitted:   381
Categorical features fitted: 49
Dropped zero-variance cols:  ['V107']

Transformed Array Dimensions:
  Train: Numerical (70000, 381), Categorical (70000, 49), Labels (70000,)
  Val:   Numerical (15000, 381), Categorical (15000, 49), Labels (15000,)
  Test:  Numerical (15000, 381), Categorical (15000, 49), Labels (15000,)


---
**3.6 PyTorch Dataset Construction (`FraudDataset`)**

We wrap the transformed arrays into `FraudDataset` instances:
- Numerical features: converted to `torch.FloatTensor` `[N, D_num]`
- Categorical features: converted to `torch.LongTensor` `[N, D_cat]` (for embedding lookup)
- Target labels: converted to `torch.FloatTensor` `[N]` (for BCEWithLogitsLoss)


In [4]:
train_dataset = FraudDataset(X_num_train, X_cat_train, y_train)
val_dataset = FraudDataset(X_num_val, X_cat_val, y_val)
test_dataset = FraudDataset(X_num_test, X_cat_test, y_test)

print("=== FraudDataset Summary ===")
print(f"Train Dataset: {len(train_dataset):,} transactions")
print(f"Val Dataset:   {len(val_dataset):,} transactions")
print(f"Test Dataset:  {len(test_dataset):,} transactions")

# Verify sample extraction
sample_num, sample_cat, sample_y = train_dataset[0]
print(f"\nSample Item [0]:")
print(f"  x_num shape: {sample_num.shape} (dtype: {sample_num.dtype})")
print(f"  x_cat shape: {sample_cat.shape} (dtype: {sample_cat.dtype})")
print(f"  y value:     {sample_y.item()} (dtype: {sample_y.dtype})")


=== FraudDataset Summary ===
Train Dataset: 70,000 transactions
Val Dataset:   15,000 transactions
Test Dataset:  15,000 transactions

Sample Item [0]:
  x_num shape: torch.Size([381]) (dtype: torch.float32)
  x_cat shape: torch.Size([49]) (dtype: torch.int64)
  y value:     0.0 (dtype: torch.float32)


---
**3.7 PyTorch DataLoader Construction**

We generate mini-batch loaders with `batch_size = 512`:
- `train_loader`: Shuffled (`shuffle=True`) for stochastic gradient updates.
- `val_loader` & `test_loader`: Sequential (`shuffle=False`) for deterministic evaluation.


In [5]:
BATCH_SIZE = 512

loaders = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=False
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

print(f"Train Loader: {len(train_loader):,} mini-batches")
print(f"Val Loader:   {len(val_loader):,} mini-batches")
print(f"Test Loader:  {len(test_loader):,} mini-batches")


2026-09-30 23:14:03,824 - INFO - Created 'train' DataLoader: 70,000 samples, 137 batches (batch_size=512)
2026-09-30 23:14:03,826 - INFO - Created 'val' DataLoader: 15,000 samples, 30 batches (batch_size=512)
2026-09-30 23:14:03,828 - INFO - Created 'test' DataLoader: 15,000 samples, 30 batches (batch_size=512)


Train Loader: 137 mini-batches
Val Loader:   30 mini-batches
Test Loader:  30 mini-batches


---
**3.8 Pipeline Verification & Quality Diagnostics**

We execute rigorous ML sanity checks across batches:
1. **Tensor Shapes**: Correct mini-batch dimensions `[B, D_num]` and `[B, D_cat]`.
2. **Data Types**: `float32` for numericals and labels; `int64` (long) for categorical indices.
3. **Missing Values**: Zero NaNs or Infs in numerical or categorical tensors.
4. **Category Bounds**: All categorical indices are valid non-negative integers $\in [0, |V|-1]$.
5. **Serialization Test**: Verified that saving and loading `TabularPreprocessor` yields identical transformations.


In [6]:
# 1. Fetch single mini-batch
batch_num, batch_cat, batch_y = next(iter(train_loader))

print("=== Mini-Batch Diagnostics ===")
print(f"Batch Numerical Shape:   {batch_num.shape} (dtype: {batch_num.dtype})")
print(f"Batch Categorical Shape: {batch_cat.shape} (dtype: {batch_cat.dtype})")
print(f"Batch Labels Shape:      {batch_y.shape} (dtype: {batch_y.dtype})")

# 2. Sanity Checks
assert not torch.isnan(batch_num).any(), "Sanity Check Failed: NaNs found in numerical batch!"
assert not torch.isinf(batch_num).any(), "Sanity Check Failed: Infs found in numerical batch!"
assert (batch_cat >= 0).all(), "Sanity Check Failed: Negative index in categorical batch!"

# Verify categorical indices are within vocabulary bounds for all features
for col_idx, col_name in enumerate(preprocessor.fitted_categorical_cols_):
    max_idx = preprocessor.vocab_sizes_[col_name]
    observed_max = batch_cat[:, col_idx].max().item()
    assert observed_max < max_idx, f"Index out of bounds for {col_name}: observed {observed_max} >= {max_idx}"

print("\nAll Sanity Checks Passed:")
print("  - Zero NaNs or Infs")
print("  - Dtypes: Float32 (num), Long (cat), Float32 (labels)")
print("  - Categorical indices strictly within [0, |V|-1]")

# 3. Save preprocessor artifact to models/
model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)
artifact_path = model_dir / "tabular_preprocessor.joblib"
preprocessor.save(artifact_path)
print(f"\nPreprocessor artifact successfully persisted to: {artifact_path}")


=== Mini-Batch Diagnostics ===
Batch Numerical Shape:   torch.Size([512, 381]) (dtype: torch.float32)
Batch Categorical Shape: torch.Size([512, 49]) (dtype: torch.int64)
Batch Labels Shape:      torch.Size([512]) (dtype: torch.float32)

All Sanity Checks Passed:
  - Zero NaNs or Infs
  - Dtypes: Float32 (num), Long (cat), Float32 (labels)
  - Categorical indices strictly within [0, |V|-1]


2026-09-30 23:14:04,053 - INFO - Saved TabularPreprocessor artifact to: C:\Ongoing Projects\2026.09.30 Multimodal-Financial-Fraud-Detector\Multimodal-Financial-Fraud-Detector\models\tabular_preprocessor.joblib



Preprocessor artifact successfully persisted to: C:\Ongoing Projects\2026.09.30 Multimodal-Financial-Fraud-Detector\Multimodal-Financial-Fraud-Detector\models\tabular_preprocessor.joblib
